In [ ]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline 

In [ ]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
# build the voacabulary of characters and mapping to/from integers 
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

In [ ]:
block_size = 3  # context length: how many characters do we take to predict the next one
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]   # crop and append 
X = torch.tensor(X)
Y = torch.tensor(Y)



In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

In [ ]:
g = torch.Generator().manual_seed(2147483647) 
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters) # no of parameters

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
lri = []
lossi =[]
for i in range(1000):

    # minibatch construct 
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass 
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 +b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])
    
    
    # backward pass 
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # update 
    lr = lrs[i]
    for p in parameters:
        p.data += -lr * p.grad 

    # track 
    lri.append(lre[i])
    lossi.append(loss.item())
    
print(loss.item())

In [ ]:
plt.plot(lri, lossi)
plt.show()

In [ ]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 +b1)

logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

In [ ]:
prob.shape

In [ ]:
 torch.randint(0, X.shape[0], (32,))